# CodeBERT HTML 特征提取

**学习目标**：用预训练 CodeBERT 提取 HTML 源码中每个 DOM 节点的特征向量，输出代码特征序列 `[N, 768]`。

完成本章后，你将拥有：
- 视觉特征序列 `[196, 768]`（第一章产出）
- 代码特征序列 `[N, 768]`（本章产出）

这两个序列是第三章「跨模态交叉注意力对齐」的输入。

**本地模型路径**（git clone 后）：`~/.cache/huggingface/hub/codebert-base`

## Part 1：Tokenizer —— 模型如何「读」HTML

CodeBERT 是微软训练的代码预训练模型，支持 Python/JS/HTML/CSS 等多种代码语言。

Tokenizer 负责把文本切分成 token id，是文本进入模型的第一步。

**任务**：加载 Tokenizer，对一段 HTML 做 tokenize，打印出 tokens 和对应的 id。

```python
from transformers import AutoTokenizer

# 用本地路径加载（git clone 后的路径）
MODEL_PATH = '/Users/didi/.cache/huggingface/hub/codebert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

sample_html = '<button class="btn-primary" style="color:red">提交</button>'

# 试试 tokenize() 和 encode() 的区别：
# tokenize() → 返回 token 字符串列表
# encode()   → 返回 token id 列表（含 [CLS] 和 [SEP]）
```

**思考**：为什么 token 数量会比单词数量多？`[CLS]` 和 `[SEP]` 是什么？

In [ ]:
# 在这里写代码


## Part 2：提取单个 DOM 节点的特征向量

和 ViT 类似，CodeBERT 输出 `last_hidden_state` 形状为 `[1, token数, 768]`。
取第 0 个位置（`[CLS]` token）的特征，作为整段输入的「摘要」向量。

**任务**：加载 CodeBERT 模型，对一个 DOM 节点提取特征向量。

```python
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(MODEL_PATH, local_files_only=True)
model.eval()

# 把节点信息序列化成字符串：标签 + 属性 + 样式
node_text = '<button class="btn-primary" style="color:red; width:120px; height:40px"></button>'

# tokenize → 转 tensor → 送入模型 → 取 CLS 特征
# 输出形状应为 [1, 768]
```

**注意**：
- 加 `with torch.no_grad():` 节省内存
- `truncation=True, max_length=128` 防止超长输入报错

In [ ]:
# 在这里写代码


## Part 3：用 BeautifulSoup 解析 HTML，提取所有节点

真实网页有几十到几百个 DOM 节点，需要自动解析出所有有意义的节点。

**任务**：用 BeautifulSoup 解析下面这段 HTML，提取所有可见节点的标签、属性、样式信息。

```python
from bs4 import BeautifulSoup

html = """
<html><body>
  <nav style="background:#333; height:60px">
    <a href="/" style="color:white; font-size:18px">Logo</a>
  </nav>
  <main style="padding:20px">
    <img src="hero.jpg" style="width:600px; height:300px"/>
    <h1 style="font-size:32px; color:#333">欢迎</h1>
    <button style="background:#1890ff; color:white; width:120px; height:40px">立即使用</button>
  </main>
</body></html>
"""

# 提示：soup.find_all(True) 遍历所有标签
# 跳过无视觉意义的标签：html, head, script, style, meta, link
# 每个节点提取：tag.name, tag.attrs, tag.get('style', '')
```

**预期输出**：打印出每个节点的标签名和样式，共 5 个节点（body/nav/a/main/img/h1/button 中去掉 html 和 body 以外的无关节点）。

In [ ]:
# 在这里写代码


## Part 4：批量编码所有节点 → 得到代码特征序列

对节点列表批量提取特征，这是论文数据管线的核心操作。

**任务**：把 Part 3 得到的节点列表，一次性批量送入 CodeBERT，输出形状 `[N, 768]` 的特征序列。

```python
# tokenizer 支持批量输入：直接传入字符串列表
# 关键参数：padding=True（对齐长度）, truncation=True, return_tensors='pt'

node_texts = [...]  # 把每个节点序列化为字符串的列表

batch_inputs = tokenizer(
    node_texts,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=128
)
# 输出形状应为 [N, 768]
```

**思考**：`padding=True` 做了什么？为什么批量处理需要它？

In [ ]:
# 在这里写代码


## Part 5：节点间余弦相似度 —— 交叉注意力的预热

交叉注意力的核心是两个向量的相似度计算。在实现完整的注意力机制之前，先用余弦相似度感受一下「两个节点的特征有多像」。

**任务**：计算所有节点特征两两之间的余弦相似度，打印相似度矩阵。

```python
from sklearn.metrics.pairwise import cosine_similarity

# node_features: [N, 768] tensor → 转成 numpy → 计算相似度
sim_matrix = cosine_similarity(...)  # 输出 [N, N]
```

**观察**：
- 对角线应全为 1.0（自己和自己完全相似）
- `<h1>` 和 `<button>` 的相似度，和 `<h1>` 和 `<a>` 的相似度，哪个更高？为什么？

In [ ]:
# 在这里写代码


## Part 6：双模态特征汇总

**任务**：把第一章的 ViT 代码和本章代码整合，打印出两个特征序列的形状，确认维度对齐。

```
视觉特征 (ViT 输出):     [1, 196, 768]  → 196 个图像 patch
代码特征 (CodeBERT 输出): [1, N,   768]  → N 个 DOM 节点
```

两个序列的最后一维都是 768，这是设计好的——下一章交叉注意力就用这个维度做点积相似度。

**完成标志**：能打印出以上两行，且 768 维度一致，本章学习目标达成。

In [ ]:
# 在这里写代码
